# ETL — Dimensión Fecha (`dim_fecha`)

Este notebook extrae las fechas únicas de publicación desde la capa Silver (`tiktok_data_eng.silver.silver_tiktok`),
deriva los atributos de calendario (año, mes, nombre de mes, día, día de semana, trimestre, fin de semana, semana del año)
y ejecuta un `MERGE` idempotente en `tiktok_data_eng.gold.dim_fecha` generando la clave subrogada incremental `fecha_id`.

In [0]:
%sql
-- Inserción / Actualización idempotente (MERGE) en dim_fecha utilizando CTE

WITH fechas_unicas AS (
    SELECT DISTINCT 
        TO_DATE(created_at) AS fecha
    FROM tiktok_data_eng.silver.silver_tiktok
    WHERE created_at IS NOT NULL
)
MERGE INTO tiktok_data_eng.gold.dim_fecha AS target
USING (
    SELECT 
        fecha,
        YEAR(fecha) AS anio,
        MONTH(fecha) AS mes,
        DATE_FORMAT(fecha, 'MMMM') AS nombre_mes,
        DAY(fecha) AS dia,
        DAYOFWEEK(fecha) AS dia_semana,
        DATE_FORMAT(fecha, 'EEEE') AS nombre_dia,
        QUARTER(fecha) AS trimestre,
        CASE WHEN DAYOFWEEK(fecha) IN (1, 7) THEN TRUE ELSE FALSE END AS es_fin_de_semana,
        WEEKOFYEAR(fecha) AS semana_anio
    FROM fechas_unicas
) AS source
ON target.fecha = source.fecha
WHEN NOT MATCHED THEN INSERT (
    fecha,
    anio,
    mes,
    nombre_mes,
    dia,
    dia_semana,
    nombre_dia,
    trimestre,
    es_fin_de_semana,
    semana_anio,
    _created_at
) VALUES (
    source.fecha,
    source.anio,
    source.mes,
    source.nombre_mes,
    source.dia,
    source.dia_semana,
    source.nombre_dia,
    source.trimestre,
    source.es_fin_de_semana,
    source.semana_anio,
    CURRENT_TIMESTAMP()
);

In [0]:
%sql
-- Validación de calidad y volumetría en dim_fecha
SELECT 
    COUNT(*) AS total_filas,
    COUNT(DISTINCT fecha_id) AS total_ids_subrogados,
    COUNT(DISTINCT fecha) AS total_fechas_unicas,
    MIN(fecha) AS fecha_minima,
    MAX(fecha) AS fecha_maxima,
    SUM(CASE WHEN fecha_id IS NULL THEN 1 ELSE 0 END) AS nulos_pk,
    COUNT(*) - COUNT(DISTINCT fecha) AS duplicados
FROM tiktok_data_eng.gold.dim_fecha;